## Class and object review
* Object creation and namespaces

In [65]:
class Student:  # equivalent to class Student(object):
    school = "USC"  # class attribute

    def __init__(self, name, major):  # instance initializer
        self.name = name              # instance attribute
        self.major = major            # instance attribute

    def introduce(self):              # ordinary instance method
        print(f"I am {self.name}. I major in {self.major}.")

# Create two instances of Student.
# Student(...) creates an instance and then calls __init__
# to initialize that instance.
david = Student("David", "CS")
john = Student("John", "ADS")

# Both methods, __init__ and introduce, receive the instance as self:
# * __init__ is a special instance method that initializes a new instance.
# * introduce is an ordinary instance method.

In [71]:
#david.introduce()
#'introduce' in Student.__dict__
Student.__dict__['introduce'](david)

I am David. I major in CS.


In [3]:
# We will cover __new__, *args, and **kwargs in more detail next week.

class Student(object):
    def __new__(cls, *args, **kwargs):
        # cls is the class currently being instantiated:
        #     Student for Student(...),
        #     TA for TA(...) where TA is a subclass of Student (see below).
        print("In __new__, cls is", cls)
        print("args =", args, "kwargs =", kwargs)

        obj = super().__new__(cls)

        print("MRO =", cls.__mro__)
        print("Current instance namespace:", obj.__dict__)
        print("Type of instance created:", type(obj))

        return obj

    def __init__(self, name, major):
        print("In __init__, self is", self)

        self.name = name
        self.major = major


david = Student(name="David", major="CS")
david

In __new__, cls is <class '__main__.Student'>
args = () kwargs = {'name': 'David', 'major': 'CS'}
MRO = (<class '__main__.Student'>, <class 'object'>)
Current instance namespace: {}
Type of instance created: <class '__main__.Student'>
In __init__, self is <__main__.Student object at 0x111151550>


In [ ]:
david = Student("David", "CS")

# Simplified conceptual steps performed behind the scenes:

# 1. Create an uninitialized Student instance.
obj = Student.__new__(Student)

# 2. Initialize that instance.
Student.__init__(obj, "David", "CS")

# 3. Return the initialized instance.
david = obj

In [ ]:
# Because Student does not define __new__, it inherits object.__new__:
Student.__new__ is object.__new__

True

In [20]:
# Each Student instance has its own instance namespace.
# It contains attributes stored directly on that instance.
david.__dict__, john.__dict__

({'name': 'David', 'major': 'CS'}, {'name': 'John', 'major': 'ADS'})

In [16]:
# Every class has a namespace that maps attribute names to objects.
#
# Student.__dict__ returns a read-only, live view of the
# Student class namespace. Its type is mappingproxy.

namespace = Student.__dict__

type(namespace)  # mappingproxy

# you can not modify the above namespace directly
# namespace['school'] = 'UCLA' # this will produce a TypeError:
#      'mappingproxy' object does not support item assignment

mappingproxy

In [17]:
# instead, modify the class
Student.school = "UCLA"

# namespace will reflect the change immediately, since it is a live view
namespace['school']

'UCLA'

## Attribute lookup

In [ ]:
# Attribute lookup through an instance can find attributes
# stored in the instance's class.

david.school       # finds Student.school
david.introduce()  # finds Student.introduce and binds david as self
john.introduce()   # finds Student.introduce and binds john as self

I am David. I major in CS.
I am John. I major in ADS.


In [28]:
david.__dict__

{'name': 'David', 'major': 'CS'}

In [30]:
type(david),

(__main__.Student,)

In [31]:
Student.__dict__['school']

'UCLA'

In [32]:
david.school = "Caltech"
david.__dict__

{'name': 'David', 'major': 'CS', 'school': 'Caltech'}

In [33]:
david.school

'Caltech'

In [34]:
type(david) is Student

True

In [35]:
Student.__mro__

(__main__.Student, object)

In [ ]:
david.introduce()

#
# 1. Check david's instance namespace.
# 2. "introduce" is not found.
# 3. Search Student's namespace.
# 4. Find the introduce function.
#    f = Student.__dict__["introduce"]
# 5. Call the function using david as self.
#    f(david)

# We will explain later how Python automatically supplies
# david as self when the function is accessed through david.

I am David. I major in CS.


In [ ]:
f = Student.__dict__["introduce"]
print(type(f))

<class 'function'>


In [ ]:
f(david)

I am David. I major in CS.


## Inheritance and MRO

In [4]:
class TA(Student): # TA is a direct subclass of Student
    pass

# TA inherits access to Student.__new__ and Student.__init__:
mary = TA('Mary', 'Math')

# The above call behaves conceptually like:
#mary = Student.__new__(TA, "Mary", "Math")
#Student.__init__(mary, "Mary", "Math")

In __new__, cls is <class '__main__.TA'>
args = ('Mary', 'Math') kwargs = {}
MRO = (<class '__main__.TA'>, <class '__main__.Student'>, <class 'object'>)
Current instance namespace: {}
Type of instance created: <class '__main__.TA'>
In __init__, self is <__main__.TA object at 0x111152a50>


In [ ]:
TA.__mro__ # (TA, Student, object)

In [26]:
type(Student) is type

True

In [25]:
# Student.__init__ initializes mary,
# so she receives her own instance attributes
mary.__dict__

{'name': 'Mary', 'major': 'Math'}

In [ ]:
# TA also inherits access to Student.introduce:
mary.introduce()

I am Mary. I major in Math.


In [ ]:
# The inherited attributes are not copied into TA.__dict__:
"__init__" in TA.__dict__    # False
"introduce" in TA.__dict__   # False

"__init__" in Student.__dict__    # True
"introduce" in Student.__dict__   # True

True

In [ ]:
# Python finds them by following the MRO
TA.__mro__
# (TA, Student, object)

(__main__.TA, __main__.Student, object)

In [ ]:
mary.introduce()

I am Mary. I major in Math.


In [ ]:
# Conceptually:
# Python finds introduce in Student and calls it with mary as self:
Student.introduce(mary)

I am Mary. I major in Math.


## Class `object` and base classes

- `object` is the root of the ordinary Python class-inheritance hierarchy and has no base class.
- `C.__bases__` contains the base classes of `C`.
- A **base class** is a direct or immediate superclass.
- An **ancestor** is any direct or indirect superclass.
- Every Python 3 class is either object itself or inherits from object directly or indirectly.
- `issubclass(C, B)` tests whether `C` inherits from `B`, directly or indirectly.
- But note that Python treats `issubclass()` as reflexive:
    * `issubclass(C, C)` is `True`, although `C` is not considered its own base class or ancestor.

In [47]:
object.__bases__, Student.__bases__, Student.__base__
issubclass(Student, object)
issubclass(Student, Student)
Student.__bases__

(object,)

In [ ]:
Student.__bases__, object.__bases__

((__main__.Person,), ())

In [ ]:
issubclass(Student, Student) # True

True

In [8]:
'Student' in Student.__bases__ # False

False

## MRO
* The MRO of a class is the ordered sequence of classes Python searches for attributes.
* It begins with the class itself, includes its direct and indirect base classes, and ends with object.
* Despite the name “method resolution order,” it applies to class attributes generally, not only methods.
* With multiple inheritance, Python uses C3 linearization to construct a consistent MRO rather than simply listing the base classes arbitrarily.

In [ ]:
class Person:
    pass

class Student(Person):
    pass

class TA(Student):
    pass

TA.__mro__    # all classes in attribute-search order
# (TA, Student, Person, object)

(__main__.TA, __main__.Student, __main__.Person, object)

In [ ]:
# For TA:
# * Student is its direct base class or immediate superclass.
# * Person is an indirect superclass or ancestor.
# * Both appear in its MRO.

In [10]:
TA.__bases__  # (Student,) — direct base classes only

(__main__.Student,)

## Diamond inheritance

In [58]:
class Person:
    pass

class Student(Person):
    pass

class Employee(Person):
    pass

class TA(Student, Employee):
    pass

# Student and Employee are:
# * TA’s direct base classes
# * TA’s immediate superclasses

# Person is an indirect superclass or ancestor of TA.

In [60]:
TA.__bases__, TA.__base__

((__main__.Student, __main__.Employee), __main__.Student)

In [ ]:
# Inheritance structure:
#        Person
#       /      \
# Student      Employee
#       \      /
#          TA

In [ ]:
# This structure forms a diamond because TA can reach Person
# through two inheritance paths:
#  TA → Student → Person
#  TA → Employee → Person

In [ ]:
# The diamond creates two questions:
# 1. If Student and Employee define the same attribute, which one should Python use?
# 2. How can Python avoid processing the shared superclass twice?

### 1. Ambiguous attribute lookup

In [62]:
class Person:
    pass

class Student(Person):
    role = "student"

class Employee(Person):
    role = "employee"

class TA(Student, Employee):
    pass

# Both direct base classes define role.
# Python resolves the ambiguity using the MRO:
TA.__mro__, TA.role

((__main__.TA, __main__.Employee, __main__.Student, __main__.Person, object),
 'employee')

In [ ]:
# Changing the order of the bases changes the MRO:
class TA(Employee, Student):
    pass

TA.__mro__, TA.role

((__main__.TA, __main__.Employee, __main__.Student, __main__.Person, object),
 'employee')

### 2. Calling shared ancestors more than once

In [31]:
class Person:
    counter = 0

    def __init__(self):
        self.counter = self.counter + 1

#p = Person()
#p.counter
p = Person()
p.counter, Person.counter

(1, 0)

In [34]:
# Consider directly calling both superclass initializers:

class Person:
    # Counts how many times Person.__init__ has been called.
    counter = 0

    def __init__(self):
        print('Person initialization')

        # note the difference from self.counter += 1
        Person.counter += 1

class Student(Person):
    def __init__(self):
        print("Student initialization")
        Person.__init__(self)

class Employee(Person):
    def __init__(self):
        print("Employee initialization")
        Person.__init__(self)

class TA(Student, Employee):
    def __init__(self):
        Student.__init__(self)
        Employee.__init__(self)

ta = TA()
ta.counter

Student initialization
Person initialization
Employee initialization
Person initialization


2

In [ ]:
s = Student()

Student initialization
Person initialization


In [ ]:
e = Employee()

Employee initialization
Person initialization


In [ ]:
ta = TA()

Student initialization
Person initialization
Employee initialization
Person initialization


In [ ]:
# Student and Employee work correctly when used separately.
#
# Trouble appears when TA combines both inheritance branches:
#   * Following only the Student branch skips Employee.__init__.
#   * Explicitly following both branches calls Person.__init__ twice.
#
# Cooperative super() solves this by following TA's MRO and
# visiting each class once.

In [ ]:
TA.__mro__

(__main__.TA, __main__.Employee, __main__.Student, __main__.Person, object)

## Cooperative inheritance using super()

In [36]:
class Person:
    counter = 0

    def __init__(self): # self is ta!
        print("Person initialization")

        # Equivalent starting point:
        #     super(Person, self).__init__()
        #
        # Begin searching after Person in type(self).__mro__.
        # Find object.__init__ and call it with the same self.
        #
        # Conceptually, the resolved call is:
        #     object.__init__(self)
        super().__init__()
        Person.counter += 1

class Student(Person):
    def __init__(self):
        print("Student initialization")

        # Equivalent starting point:
        #     super(Student, self).__init__()
        #
        # Begin searching after Student in type(self).__mro__.
        # Find Employee.__init__ and call it with the same self.
        #
        # Conceptually, the resolved call is:
        #     Employee.__init__(self)
        super().__init__()

class Employee(Person):
    def __init__(self): # self refers to ta
        print("Employee initialization")

        # Equivalent starting point:
        #     super(Employee, self).__init__()
        #
        # Begin searching after Employee in type(self).__mro__.
        # Find Person.__init__ and call it with the same self.
        #
        # Conceptually, the resolved call is:
        #     Person.__init__(self)
        super().__init__()


class TA(Student, Employee):
    def __init__(self):
        print("TA initialization")

        # Equivalent starting point:
        #     super(TA, self).__init__()
        #
        # Begin searching after TA in type(self).__mro__.
        # Find Student.__init__ and call it with the same self.
        #
        # Conceptually, the resolved call is:
        #     Student.__init__(self)
        super(TA, self).__init__()


# The comments above trace what happens behind the scenes
# while evaluating TA().
#
# Throughout the initialization chain:
#   * self refers to the same newly created TA instance.
#   * type(self) is TA.
#   * attribute lookup follows TA.__mro__:
#
#       TA → Student → Employee → Person → object
#
# The name ta is assigned only after initialization finishes.

ta = TA()

# ta = object.__new__(TA) # ta uniniatialized
# TA.__init__(ta) # ta initialized

ta.counter

TA initialization
Student initialization
Employee initialization
Person initialization


1

In [81]:
print(type(ta))

<class '__main__.TA'>


In [79]:
TA.__mro__

(__main__.TA, __main__.Student, __main__.Employee, __main__.Person, object)

In [ ]:
# A particularly important point is that this call inside Student:
# super().__init__()

# does not necessarily call Person.__init__ directly.
# For a TA instance, it behaves like:
super(Student, ta).__init__()

# The class following Student in TA.__mro__ is Employee.
# Therefore, it calls Employee.__init__.

# Each class cooperatively calls the next class in the MRO,
# so Person.__init__ runs exactly once.

Employee initialization
Person initialization


## Mixins
* A mixin is a small class that provides a reusable capability to other classes.
* Mixins usually:
    - Are not intended to be instantiated by themselves.
    - Provide a small set of related methods.
    - Do not represent the main identity of an object.
    - Often have names ending in Mixin.
    

In [ ]:
class ToDictMixin:
    def to_dict(self):
        # Return a copy of the instance namespace.
        return self.__dict__.copy()

class DisplayMixin:
    def display(self):
        print(type(self).__name__)

        for name, value in self.__dict__.items():
            print(f"  {name}: {value}")

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

class Student(ToDictMixin, DisplayMixin, Person):
    def __init__(self, name, major):
        # super() searches through the MRO until it finds
        # Person.__init__.
        # Note neither mixin defines __init__.
        super().__init__(name)
        self.major = major

In [ ]:
david = Student("David", "CS")

In [ ]:
david.to_dict()
# {'name': 'David', 'major': 'CS'}

{'name': 'David', 'major': 'CS'}

In [ ]:
david.display()

Student
  name: David
  major: CS


In [ ]:
Student.__mro__
# (Student, ToDictMixin, DisplayMixin, Person, object)

(__main__.Student,
 __main__.ToDictMixin,
 __main__.DisplayMixin,
 __main__.Person,
 object)

In [ ]:
# Note: These mixins provide distinct methods,
# so they do not form a cooperative method chain.
# In general, a mixin uses super()
# when multiple classes contribute to the same operation.
# You will practice that pattern in HW6.